# Module 5 — Inverse Kinematics
## Unit 1 — The Inverse Problem
### Lesson 1.4 — The Inverse Problem (Unit 1 Recap)

*Physical AI Curriculum · hands-on notebook. Run **Kernel → Restart & Run All**.*


## Unit 1 synthesis
Problem statement + nonlinearity + multiplicity + reachability in one capstone check.


In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

def fk_two_link(theta1, theta2, L1, L2):
    """Forward kinematics: planar 2-link gripper position."""
    x = L1*np.cos(theta1) + L2*np.cos(theta1+theta2)
    y = L1*np.sin(theta1) + L2*np.sin(theta1+theta2)
    return np.array([x, y])

def reachable(x, y, L1, L2, tol=1e-9):
    r = np.hypot(x, y)
    return abs(L1-L2)-tol <= r <= L1+L2+tol

def ik_two_link(x, y, L1, L2):
    """Return both (theta1,theta2) solutions; [] if unreachable, one if on boundary."""
    r2 = x*x + y*y
    c2 = (r2 - L1*L1 - L2*L2)/(2*L1*L2)
    if c2 < -1-1e-9 or c2 > 1+1e-9:
        return []
    c2 = max(-1.0, min(1.0, c2))
    sols=[]
    for sign in (+1.0, -1.0):
        t2 = sign*np.arccos(c2)
        t1 = np.arctan2(y, x) - np.arctan2(L2*np.sin(t2), L1+L2*np.cos(t2))
        sols.append((t1, t2))
        if abs(np.sin(t2)) < 1e-6:    # boundary: the two coincide
            break
    return sols

L1,L2=0.4,0.3
def classify(x,y):
    if not reachable(x,y,L1,L2): return 0
    sols=ik_two_link(x,y,L1,L2); return len(sols)
print("(0.8,0):", classify(0.8,0))      # outside -> 0
print("(0.7,0):", classify(0.7,0))      # outer boundary -> 1
print("(0.4,0.2):", classify(0.4,0.2))  # inside -> 2


## Check your work


In [ ]:
assert classify(0.8,0)==0
assert classify(0.7,0)==1
assert classify(0.4,0.2)==2
# every found solution must verify under FK
for (t1,t2) in ik_two_link(0.4,0.2,L1,L2):
    assert np.allclose(fk_two_link(t1,t2,L1,L2),[0.4,0.2],atol=1e-9)
print("All checks passed.")
